In [ ]:
import os
from dotenv import load_dotenv
import mlflow
import pickle
from sklearn.model_selection import train_test_split
import pandas as pd
from catboost import CatBoostClassifier

load_dotenv("/home/mle-user/mle_projects/mle-mlflow/.env")  # загружаем .env в os.environ

os.environ["MLFLOW_S3_ENDPOINT_URL"] = "https://storage.yandexcloud.net" #endpoint бакета от YandexCloud
os.environ["AWS_ACCESS_KEY_ID"] = os.getenv("AWS_ACCESS_KEY_ID") # получаем id ключа бакета, к которому подключён MLFlow, из .env
os.environ["AWS_SECRET_ACCESS_KEY"] = os.getenv("AWS_SECRET_ACCESS_KEY") # получаем ключ бакета, к которому подключён MLFlow, из .env

# определяем глобальные переменные
# поднимаем MLflow локально
TRACKING_SERVER_HOST = "127.0.0.1"
TRACKING_SERVER_PORT = 5000

EXPERIMENT_NAME = f"sprint_test"
RUN_NAME = "test_sprint_run"

mlflow.set_tracking_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")

mlflow.set_experiment(EXPERIMENT_NAME)

df = pd.read_csv('~/mle_projects/mle-dvc/data/initial_data.csv')

with open(r"models/fitted_model.pkl", 'rb') as f:
    model = pickle.load(f)

y = df['end_date'].notna().astype(int)
X = df.drop(columns=['id', 'begin_date', 'end_date'], errors='ignore')

_, X_test, _, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

prediction = model.predict(X_test)
proba = model.predict_proba(X_test)[:, 1]

/home/mle-user/.local/lib/python3.10/site-packages/pydantic/_internal/_fields.py:151: UserWarning: Field "model_server_url" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/home/mle-user/.local/lib/python3.10/site-packages/pydantic/_internal/_config.py:322: UserWarning: Valid config keys have changed in V2:
* 'schema_extra' has been renamed to 'json_schema_extra'
  warnings.warn(message, UserWarning)
/home/mle-user/.local/lib/python3.10/site-packages/sklearn/preprocessing/_discretization.py:248: FutureWarning: In version 1.5 onwards, subsample=200_000 will be used by default. Set subsample explicitly to silence this warning in the mean time. Set subsample=None to disable subsampling explicitly.
  warnings.warn(
/home/mle-user/.local/lib/python3.10/site-packages/mlflow/models/signature.py:212: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in 

In [ ]:
import os
from dotenv import load_dotenv
import mlflow
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score,
    recall_score, confusion_matrix, log_loss
)

load_dotenv("/home/mle-user/mle_projects/mle-mlflow/.env")

os.environ["MLFLOW_S3_ENDPOINT_URL"] = "https://storage.yandexcloud.net"
os.environ["AWS_ACCESS_KEY_ID"]      = os.getenv("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"]  = os.getenv("AWS_SECRET_ACCESS_KEY")

TRACKING_SERVER_HOST = "127.0.0.1"
TRACKING_SERVER_PORT = 5000
EXPERIMENT_NAME      = "churn_prediction"
RUN_NAME             = "model_baseline"

mlflow.set_tracking_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")
mlflow.set_experiment(EXPERIMENT_NAME)

df = pd.read_csv('~/mle_projects/mle-dvc/data/initial_data.csv')

y = df['target']
X = df.drop(columns=['id', 'begin_date', 'end_date', 'target'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

cat_cols = list(X.select_dtypes(include=['object']).columns)
model = CatBoostClassifier(iterations=100, cat_features=cat_cols, verbose=0)
model.fit(X_train, y_train)

prediction = model.predict(X_test)
probas     = model.predict_proba(X_test)[:, 1]

metrics = {}

_, err1, _, err2 = confusion_matrix(y_test, prediction, normalize='all').ravel()

metrics["err1"]      = err1
metrics["err2"]      = err2
metrics["auc"]       = roc_auc_score(y_test, probas)
metrics["precision"] = precision_score(y_test, prediction)
metrics["recall"]    = recall_score(y_test, prediction)
metrics["f1"]        = f1_score(y_test, prediction)
metrics["logloss"]   = log_loss(y_test, prediction)

print(metrics)

{'err1': 0.06766381766381767, 'err2': 0.14387464387464388, 'auc': 0.843550991645062, 'precision': 0.6801346801346801, 'recall': 0.5415549597855228, 'f1': 0.6029850746268657, 'logloss': 6.828783334405386}


In [3]:
import os
import mlflow
from dotenv import load_dotenv

EXPERIMENT_NAME      = "churn_prediction_V2"
RUN_NAME             = "model_baseline_test_sprint"

REGISTRY_MODEL_NAME = "churn_model_V2"

load_dotenv("/home/mle-user/mle_projects/mle-mlflow/.env")

os.environ["MLFLOW_S3_ENDPOINT_URL"] = "https://storage.yandexcloud.net"
os.environ["AWS_ACCESS_KEY_ID"]      = os.getenv("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"]  = os.getenv("AWS_SECRET_ACCESS_KEY")

mlflow.set_tracking_uri(f"http://127.0.0.1:5000")
mlflow.set_experiment(EXPERIMENT_NAME)

pip_requirements = "/home/mle-user/mle_projects/mle-mlflow/requirements.txt"
signature = mlflow.models.infer_signature(X_test, prediction)
input_example = X_test[:10]
metadata = {'model_type': 'monthly'}


experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id

with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    run_id = run.info.run_id
    
    mlflow.log_metrics(metrics)

    model_info = mlflow.catboost.log_model(
        cb_model=model,
        artifact_path="models",
        pip_requirements=pip_requirements,
        signature=signature,
        input_example=input_example,
        metadata = metadata,
        registered_model_name=REGISTRY_MODEL_NAME,
        await_registration_for=60
    )

/home/mle-user/.local/lib/python3.10/site-packages/mlflow/models/signature.py:212: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  inputs = _infer_schema(model_input) if model_input is not None else None
Successfully registered model 'churn_model_V2'.
2026/06/10 11:17:12 INFO mlflow.tracking._model_registry.client: Waiting up to 60 seconds for model version to finish creation

In [10]:
model = mlflow.catboost.load_model(f"models:/{REGISTRY_MODEL_NAME}/latest")
model_predictions = model.predict(X_test)

assert model_predictions.dtype == int
print(model_predictions[:10])

[0 0 1 0 0 0 0 0 0 0]
